# Session 06 · Lab — From model to app: a Gradio prototype and your first Startti agent
*Introduction to Basic Application Development · Introducción al desarrollo de aplicaciones básicas*

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/06-basic-ai-applications/lab.ipynb)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/06-basic-ai-applications/lab.ipynb)

In the slides we turned a model into a *product*: a user, a job to be done, a confidence threshold, a
human in the loop and a feedback log. Today you build that product twice: once **in code** (Python +
Gradio) and once **without code** (a Startti agent you call through its API).

**Learning objectives**

1. Train a fast intent classifier and expose its confidence (RAE 3).
2. Route requests with a confidence threshold that minimises the expected cost (RAE 4).
3. Build a Gradio `Blocks` app with routing, a feedback log and a text-generation tab (RAE 3).
4. Create, publish and call a Startti agent from Kaggle, and wrap it in a chat interface (RAE 3).

**Time plan (60 min)**

| Part | Topic | Time |
|---|---|---|
| 1 | A fast intent classifier (Banking77, TF-IDF + logistic regression) | 10' |
| 2 | Confidence thresholds and human-in-the-loop routing | 12' |
| 3 | A Gradio app: router, feedback log and text generator | 18' |
| 4 | Your first Startti agent, called from Kaggle | 17' |
| — | Wrap-up | 3' |

**Kaggle checklist**

- *Settings → Internet* → **On** (we download a dataset and a small model).
- *Settings → Accelerator* → **None**: the CPU is enough today, so save your GPU quota.
- *Add-ons → Secrets* → add **`STARTTI_API_KEY`** (your `adp_…` key) and attach it to this notebook. You create
  the key in Part 4; if you do not have it yet, every Startti cell is skipped and the chat falls back to a
  Python bot. Guide: [configuracion-startti.md](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/docs/configuracion-startti.md).

In [ ]:
# --- Course setup (do not edit) ---
import os, sys, random, subprocess, importlib
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"  # used by the course smoke test
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules

def ensure(package, import_name=None):
    """Install a package only if it is missing (Kaggle already ships most of them)."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import socket
if os.environ.get("HF_HUB_OFFLINE") != "1":  # models and datasets come from the Hugging Face Hub
    try:
        socket.create_connection(("huggingface.co", 443), timeout=10).close()
    except OSError:
        raise RuntimeError(
            "No internet connection: this notebook downloads models and datasets from Hugging Face.\n"
            "Kaggle: in the right-hand panel open Settings (Session options) and switch Internet on. "
            "The switch only appears once your account is phone-verified (kaggle.com/settings). "
            "Then run this cell again. Colab has internet by default. "
            "On your own computer: check your connection, or set HF_HUB_OFFLINE=1 if the models are already cached."
        ) from None

import numpy as np
import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | Kaggle: {IN_KAGGLE} | Colab: {IN_COLAB} | Fast dev run: {FAST_DEV_RUN}")
# --- end of course setup ---
ensure("datasets")
ensure("scikit-learn", "sklearn")
ensure("gradio")

## Part 1 — A fast intent classifier (10')

**The job to be done.** Andes Bank (a fictional digital bank) receives thousands of customer messages. The
support team wants each message tagged with its *intent* so that simple ones get an instant answer and the
rest reach the right person. Input: one free-text message. Output: an intent **and how confident the model is**.

**The data.** [Banking77](https://huggingface.co/datasets/legacy-datasets/banking77) (Casanueva et al., 2020):
about 13,000 real-style customer-service queries **in English**, labelled with 77 banking intents.

**The model.** TF-IDF features + logistic regression. It is not the most accurate option (the DistilBERT you
fine-tuned in S04 is stronger), but it trains in seconds on a CPU, answers in milliseconds and gives class
probabilities through `predict_proba`. For a first prototype, start with the simplest model that could work.

In [ ]:
import time
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

banking = load_dataset("legacy-datasets/banking77")
INTENTS = banking["train"].features["label"].names
train_df = banking["train"].to_pandas()
test_df = banking["test"].to_pandas()

if FAST_DEV_RUN:  # smaller samples for the automatic test; same code path
    train_df = train_df.groupby("label", group_keys=False).head(30)
    test_df = test_df.groupby("label", group_keys=False).head(8)

# Validation split: we will choose the threshold on it (never on the test set)
X_train, X_val, y_train, y_val = train_test_split(
    train_df["text"].tolist(), train_df["label"].to_numpy(),
    test_size=0.2, stratify=train_df["label"], random_state=SEED)
X_test, y_test = test_df["text"].tolist(), test_df["label"].to_numpy()

print(f"{len(INTENTS)} intents | train {len(X_train)} · validation {len(X_val)} · test {len(X_test)}")
print("Examples:")
for text, label in zip(X_train[:3], y_train[:3]):
    print(f"  [{INTENTS[label]}] {text}")

In [ ]:
clf = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),  # words and word pairs
    LogisticRegression(C=10.0, max_iter=1000),                 # higher C = weaker regularisation
)
t0 = time.perf_counter()
clf.fit(X_train, y_train)
print(f"Training time: {time.perf_counter() - t0:.1f} s")

val_acc = (clf.predict(X_val) == y_val).mean()
test_acc = (clf.predict(X_test) == y_test).mean()
print(f"Accuracy · validation {val_acc:.3f} · test {test_acc:.3f}")


def predict_intent(text, k=3):
    """Top-k (intent, probability) pairs for one message."""
    probs = clf.predict_proba([text])[0]
    top = np.argsort(probs)[::-1][:k]
    return [(INTENTS[clf.classes_[i]], float(probs[i])) for i in top]


# Latency of one prediction, measured the way a user would feel it (one message at a time)
timings = []
for text in X_test[:100]:
    t0 = time.perf_counter()
    predict_intent(text)
    timings.append(1000 * (time.perf_counter() - t0))
print(f"Latency per message · median {np.median(timings):.1f} ms · p95 {np.percentile(timings, 95):.1f} ms")

In [ ]:
for text in ["I still haven't received my new card",
             "What's the weather like in Bogotá tomorrow?",
             "Perdí mi tarjeta, ¿qué hago?"]:
    top = predict_intent(text)
    print(f"{text!r}")
    for intent, p in top:
        print(f"   {p:6.1%}  {intent}")

**✋ Checkpoint (Zoom chat).** The weather question and the Spanish message still received an intent. Why does a
classifier *always* answer, and which number is the only warning sign the app can use?

**🧪 Try it.** Write a message the model classifies **confidently but wrongly** (for example, a question
that mixes two topics). What does that tell you about using confidence as a safety net?

<details><summary>Show a solution</summary>

A classifier is a *closed world*: it spreads probability over its 77 intents, so even an out-of-scope or
Spanish message gets the "least bad" label. Its top probability is the only signal, and it is usually lower for
those messages. A mixed message such as `"I was charged twice and my card is not working"` can still get a
fairly high probability for one intent. Confidence reduces errors but never removes them: you still need
human review, feedback and monitoring.

```python
for p_intent, p in predict_intent("I was charged twice and my card is not working"):
    print(f"{p:.1%} {p_intent}")
```
</details>

## Part 2 — Confidence thresholds and human-in-the-loop routing (12')

The routing rule from the slides: if the top probability is at least the threshold $t$, answer automatically;
otherwise send the message to a person.

- **Coverage** $c(t)$: share of messages answered automatically.
- **Accuracy of the automated part** $a(t)$: share of those automatic answers that are right.
- **Expected cost per 1,000 messages**
  $$1000\,\big[\,c\,C_{auto} + c\,(1-a)\,C_{err} + (1-c)\,C_{hum}\,\big]$$

The unit costs below are **illustrative** (USD); in a real project they come from the business (cost of a
human contact, of a wrong answer: rework, repeat contact, compensation, churn).

In [ ]:
import matplotlib.pyplot as plt

COST_AUTO = 0.05   # USD per automated answer (compute, platform) · illustrative
COST_HUMAN = 2.00  # USD per message handled by a person · illustrative
COST_ERROR = 15.0  # USD per wrong automated answer · illustrative
THRESHOLDS = np.round(np.arange(0.30, 0.91, 0.05), 2)


def routing_table(confidence, correct, thresholds, cost_auto=COST_AUTO, cost_human=COST_HUMAN,
                  cost_error=COST_ERROR):
    """Coverage, accuracy of the automated part and expected cost for each threshold."""
    rows = []
    for t in thresholds:
        auto = confidence >= t
        coverage = auto.mean()
        auto_accuracy = correct[auto].mean() if auto.any() else float("nan")
        error_rate = (auto & ~correct).mean()  # wrong automatic answers, as a share of ALL messages
        cost = 1000 * (coverage * cost_auto + error_rate * cost_error + (1 - coverage) * cost_human)
        rows.append({"threshold": float(t), "coverage": coverage, "auto_accuracy": auto_accuracy,
                     "errors_per_1000": 1000 * error_rate, "cost_per_1000": cost})
    return pd.DataFrame(rows)


P_val = clf.predict_proba(X_val)
conf_val = P_val.max(axis=1)
correct_val = clf.classes_[P_val.argmax(axis=1)] == y_val

table_val = routing_table(conf_val, correct_val, THRESHOLDS)
BEST_THRESHOLD = float(table_val.loc[table_val["cost_per_1000"].idxmin(), "threshold"])
print(table_val.round(3).to_string(index=False))
print(f"\nLowest expected cost on validation at threshold {BEST_THRESHOLD:.2f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(table_val["threshold"], table_val["cost_per_1000"], marker="o")
ax1.axhline(1000 * COST_HUMAN, ls="--", c="gray", label="everything to a human")
ax1.axvline(BEST_THRESHOLD, ls=":", c="C3", label=f"best threshold {BEST_THRESHOLD:.2f}")
ax1.set(xlabel="Threshold", ylabel="USD per 1,000 messages", title="Expected cost (validation)")
ax1.legend()
ax2.plot(table_val["threshold"], table_val["coverage"], marker="o", label="coverage (automated)")
ax2.plot(table_val["threshold"], table_val["auto_accuracy"], marker="s", label="accuracy of automated part")
ax2.set(xlabel="Threshold", ylim=(0, 1.02), title="The trade-off")
ax2.legend()
plt.tight_layout()
plt.show()

**Is a confidence of 0.8 really 80%?** If the probabilities were *calibrated*, the cheapest rule would be to
automate whenever $(1-p)\,C_{err} + C_{auto} < C_{hum}$, that is, when $p > t^* = 1 - (C_{hum} - C_{auto})/C_{err}$.
The table below compares the model's confidence with its real accuracy on the validation set.

In [ ]:
t_star = 1 - (COST_HUMAN - COST_AUTO) / COST_ERROR
calibration = pd.DataFrame({"confidence": conf_val, "correct": correct_val})
calibration["bin"] = pd.cut(calibration["confidence"], bins=[0, 0.3, 0.5, 0.7, 0.9, 1.0], include_lowest=True)
print(calibration.groupby("bin", observed=True).agg(
    messages=("correct", "size"), mean_confidence=("confidence", "mean"), accuracy=("correct", "mean")).round(3))
print(f"\nTheoretical threshold if calibrated: t* = {t_star:.2f} · empirical best: {BEST_THRESHOLD:.2f}")

# Final check on the test set, with the threshold chosen on validation
P_test = clf.predict_proba(X_test)
conf_test = P_test.max(axis=1)
correct_test = clf.classes_[P_test.argmax(axis=1)] == y_test
final = routing_table(conf_test, correct_test, [0.0, BEST_THRESHOLD])
final.insert(0, "policy", ["everything automatic", f"threshold {BEST_THRESHOLD:.2f}"])
print(final.round(3).to_string(index=False))
print(f"Everything to a human: {1000 * COST_HUMAN:.0f} USD per 1,000 messages")

**✋ Checkpoint.** (1) In the calibration table, is our model over- or under-confident? What does that do to
the theoretical threshold $t^*$? (2) Why do we choose the threshold on the **validation** set and only *report*
it on the test set? (Think of Session 05.)

**🧪 Try it.** A wrong answer about a lost card can cost far more than 15 USD. Set `COST_ERROR = 50` and re-run
the two cells above. Where does the best threshold move, and what happens to coverage?

<details><summary>Show a solution</summary>

With TF-IDF + logistic regression the accuracy in each bin is usually **higher** than the mean confidence: the
model is *under-confident*, so $t^*$ is too conservative and the empirical best threshold is lower. With a
higher error cost the best threshold moves up and coverage falls: fewer, safer automatic answers. The threshold
is chosen on validation because choosing it on test would leak test information into a decision, and the test
estimate would then be optimistic.

```python
table_50 = routing_table(conf_val, correct_val, THRESHOLDS, cost_error=50)
print(table_50.loc[table_50["cost_per_1000"].idxmin()])
```
</details>

## Part 3 — A Gradio app: router, feedback log and text generator (18')

[Gradio](https://www.gradio.app/docs) turns Python functions into a web app. With `gr.Blocks` you control the
layout: **components** (`Textbox`, `Slider`, `Button`, `Label`, `Markdown`, `Dataframe`) and **events**
(`button.click(fn, inputs, outputs)`). Our app applies four guidelines from Amershi et al. (2019):

| Guideline | Where in the app |
|---|---|
| G1 Make clear what the system can do | The description at the top |
| G2 Make clear how well it can do it | Top-3 intents with their probabilities |
| G10 Scope services when in doubt | Below the threshold → "send to a human" with suggestions |
| G15 Encourage granular feedback | 👍 / 👎 plus the correct intent, saved to a log |

First we write the functions and test them *without* the interface; then we attach them to the UI.

In [ ]:
import tempfile
from pathlib import Path

APP_DESCRIPTION = (
    "**Andes Bank · support router (prototype).** I read a customer message **in English** and suggest one of "
    "77 banking topics. When I am not confident enough, I send the message to a person. "
    "I cannot see accounts or make changes.")
AUTO_REPLY = "✅ **Auto-answer** · topic `{intent}` · confidence {conf:.0%} (threshold {threshold:.0%})"
HANDOFF_REPLY = ("🙋 **Send to a human** · best guess `{intent}` ({conf:.0%}) is below the {threshold:.0%} "
                 "threshold. Suggestions for the advisor: {suggestions}")

INTERACTIONS = []  # every routed message: our usage log
FEEDBACK = []      # every 👍 / 👎: our feedback log
FEEDBACK_COLUMNS = ["time", "text", "intent", "confidence", "threshold", "decision", "latency_ms",
                    "verdict", "corrected_intent"]
LOG_PATH = Path(tempfile.gettempdir()) / "s06_feedback_log.csv"  # a temporary file, outside the project


def route_message(text, threshold=BEST_THRESHOLD):
    """Classify one message and decide: automatic answer or human."""
    text = (text or "").strip()
    if not text:
        return {}, "Type a customer message first.", None
    t0 = time.perf_counter()
    top = predict_intent(text, k=3)
    latency_ms = 1000 * (time.perf_counter() - t0)
    intent, conf = top[0]
    if conf >= threshold:
        decision, message = "auto", AUTO_REPLY.format(intent=intent, conf=conf, threshold=threshold)
    else:
        suggestions = ", ".join(f"`{name}`" for name, _ in top)
        decision = "human"
        message = HANDOFF_REPLY.format(intent=intent, conf=conf, threshold=threshold, suggestions=suggestions)
    record = {"time": time.strftime("%H:%M:%S"), "text": text, "intent": intent,
              "confidence": round(conf, 4), "threshold": float(threshold), "decision": decision,
              "latency_ms": round(latency_ms, 1)}
    INTERACTIONS.append(record)
    return {name: p for name, p in top}, message, record


def log_feedback(record, verdict, corrected_intent=""):
    """Append a 👍 / 👎 verdict about the last routed message and save the log as CSV."""
    if not record:
        return pd.DataFrame(FEEDBACK, columns=FEEDBACK_COLUMNS), "Route a message first, then give feedback."
    row = {**record, "verdict": verdict, "corrected_intent": corrected_intent if verdict == "wrong" else ""}
    FEEDBACK.append(row)
    table = pd.DataFrame(FEEDBACK, columns=FEEDBACK_COLUMNS)
    table.to_csv(LOG_PATH, index=False)
    return table, f"Saved feedback #{len(FEEDBACK)} to `{LOG_PATH}`."


# Test the functions directly (no interface needed)
for text in ["I still haven't received my new card", "Can you recommend a good restaurant?"]:
    labels, message, record = route_message(text)
    print(message)
table, status = log_feedback(record, "wrong", "lost_or_stolen_card")
print(status)

The second tab is a small **text generator** with `SmolLM2-135M-Instruct` (the model from S01), so you can
feel the difference in latency between a classifier (milliseconds) and a generative model (seconds).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

GEN_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
gen_tok = AutoTokenizer.from_pretrained(GEN_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(GEN_NAME).float().to(DEVICE)
gen_model.eval()
MAX_NEW_TOKENS = 12 if FAST_DEV_RUN else 120
SYSTEM_PROMPT = "You write short, polite customer-service replies for Andes Bank, a fictional digital bank."


def generate_text(prompt, temperature=0.7, max_new_tokens=MAX_NEW_TOKENS):
    """Generate a reply with SmolLM2; temperature 0 means greedy decoding."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
    inputs = gen_tok.apply_chat_template(messages, add_generation_prompt=True,
                                         return_tensors="pt", return_dict=True).to(DEVICE)
    kwargs = {"max_new_tokens": int(max_new_tokens), "pad_token_id": gen_tok.eos_token_id}
    if temperature > 0:
        kwargs.update(do_sample=True, temperature=float(temperature), top_p=0.95)
    else:
        kwargs.update(do_sample=False)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = gen_model.generate(**inputs, **kwargs)
    seconds = time.perf_counter() - t0
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    reply = gen_tok.decode(new_tokens, skip_special_tokens=True).strip()
    return reply, f"⏱️ {seconds:.1f} s · {len(new_tokens)} new tokens"


reply, info = generate_text("Write a two-sentence reply to a customer whose new card has not arrived yet.")
print(reply, "\n", info)

Now the interface. Read the code top to bottom: layout first (`Tab`, `Row`, `Column`), then the events that
connect buttons to the functions you just tested. `gr.State` keeps the last routed message so the feedback
buttons know what they refer to.

In [ ]:
import gradio as gr

with gr.Blocks(title="S06 · Andes Bank support prototype") as demo:
    gr.Markdown("# Andes Bank · support prototype\n" + APP_DESCRIPTION)
    with gr.Tab("Intent router"):
        with gr.Row():
            with gr.Column():
                msg_in = gr.Textbox(label="Customer message (English)", lines=3,
                                    placeholder="e.g. My card was declined at the supermarket")
                thr_in = gr.Slider(0.0, 1.0, value=BEST_THRESHOLD, step=0.05, label="Confidence threshold")
                route_btn = gr.Button("Route message", variant="primary")
            with gr.Column():
                top3_out = gr.Label(num_top_classes=3, label="Top-3 intents")
                decision_out = gr.Markdown()
        last_record = gr.State(None)
        gr.Markdown("### Was the routing right?")
        with gr.Row():
            correct_in = gr.Dropdown(choices=INTENTS, label="If wrong: the correct intent")
            good_btn = gr.Button("👍 Right")
            bad_btn = gr.Button("👎 Wrong")
        feedback_status = gr.Markdown()
        feedback_table = gr.Dataframe(value=pd.DataFrame(columns=FEEDBACK_COLUMNS), interactive=False,
                                      label="Feedback log")
        route_btn.click(route_message, inputs=[msg_in, thr_in], outputs=[top3_out, decision_out, last_record])
        msg_in.submit(route_message, inputs=[msg_in, thr_in], outputs=[top3_out, decision_out, last_record])
        good_btn.click(lambda r: log_feedback(r, "right"), inputs=[last_record],
                       outputs=[feedback_table, feedback_status])
        bad_btn.click(lambda r, c: log_feedback(r, "wrong", c), inputs=[last_record, correct_in],
                      outputs=[feedback_table, feedback_status])
    with gr.Tab("Text generator"):
        prompt_in = gr.Textbox(label="Instruction", lines=3,
                               value="Write a two-sentence reply to a customer whose new card has not arrived yet.")
        temp_in = gr.Slider(0.0, 1.5, value=0.7, step=0.1, label="Temperature (0 = greedy)")
        len_in = gr.Slider(16, 256, value=max(16, MAX_NEW_TOKENS), step=8, label="Max new tokens")
        gen_btn = gr.Button("Generate", variant="primary")
        gen_out = gr.Textbox(label="Model output", lines=6)
        gen_info = gr.Markdown()
        gen_btn.click(generate_text, inputs=[prompt_in, temp_in, len_in], outputs=[gen_out, gen_info])

if FAST_DEV_RUN:
    print("Fast dev run: the app is built but not launched.")
else:
    demo.launch(share=True)  # prints a temporary public link; the app also appears below the cell

Use the app for 3–4 minutes: route at least five messages (include one ambiguous and one out-of-scope), move the
threshold slider, and give feedback on each. Then run the next cell to see what your logs captured.
When you finish, run `demo.close()` to stop the app.

In [ ]:
print(f"{len(INTERACTIONS)} routed messages · {len(FEEDBACK)} feedback entries")
display(pd.DataFrame(INTERACTIONS).tail(10))
display(pd.DataFrame(FEEDBACK, columns=FEEDBACK_COLUMNS).tail(10))

**✋ Checkpoint.** Which of Amershi's 18 guidelines does this app *not* apply yet? Name one and say how you
would add it (for example, G11 "make clear why the system did what it did").

**🧪 Try it.** The feedback log is only useful if someone reads it. Compute, from `FEEDBACK`, the share of
👎 among automatic answers and the most frequent corrected intent.

<details><summary>Show a solution</summary>

```python
fb = pd.DataFrame(FEEDBACK, columns=FEEDBACK_COLUMNS)
auto = fb[fb["decision"] == "auto"]
if len(auto):
    print(f"Wrong automatic answers: {(auto['verdict'] == 'wrong').mean():.0%} of {len(auto)}")
print(fb.loc[fb["verdict"] == "wrong", "corrected_intent"].value_counts().head(3))
```

Corrected examples are new labelled data: in Session 11 you will use logs like this one to retrain and to
monitor the model. Remember that real logs contain personal data (Ley 1581 de 2012): minimise, anonymise and
define how long you keep them.
</details>

## Part 4 — Your first Startti agent, called from Kaggle (17')

Now the same assistant **without writing model code**. In [Startti ADP](https://adp.startti.ai) you build
agents on a visual canvas (executed by the Colmena engine): you connect nodes such as a chat input, a
language-model call and a chat output, and you can add knowledge bases, tools/APIs and sub-agents. When the
agent works in the preview, you **publish** it and call it by API.

Follow the course guide [configuracion-startti.md](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/docs/configuracion-startti.md)
(sections 2–7) and the [official documentation](https://adp.startti.ai/docs). Interfaces change: if a button
does not have exactly the name used here, look for the equivalent option.

1. **Sign in** at [app.startti.ai](https://app.startti.ai) with the course licence and open your workspace.
2. **Create a project**, for example `S06 - Andes Bank assistant`.
3. **Build the minimum flow** on the canvas: a *chat input* node → a *language-model call* node (choose the
   model and paste the system instructions below) → a *chat output* node.
4. **Test it in the preview** with three messages: one in scope, one out of scope and one in Spanish.
5. **Publish** the agent. Publishing is an explicit step, different from saving; the API answers **409** while
   the agent is not published.
6. **Create an API key** in the ADP backoffice (the API keys option of your workspace). It starts with `adp_`
   and is **shown only once**: store it right away in *Kaggle → Add-ons → Secrets* as `STARTTI_API_KEY`,
   attach the secret to this notebook and re-run the client cell below.
7. **Copy the agent ID** (it appears on the agent's page and in its URL) into `AGENT_ID`.

**System instructions (starting template).** The guide has a Spanish version; the agent can answer in the
customer's language.

```text
You are the virtual assistant of Andes Bank, a fictional digital bank.
Goal: answer frequent questions about cards, transfers and support channels.
Tone: friendly, clear and brief (at most 4 sentences). Answer in the customer's language.
Rules:
- Say that you are a virtual assistant (AI). Never claim to be a person.
- If you do not know the answer or the topic is out of scope, say so and offer a human advisor.
- Lost, stolen or compromised cards and unrecognised payments: hand off to a human advisor immediately.
- Never ask for passwords, full card numbers or identity documents.
- Do not invent fees, deadlines or policies.
```

🔐 **Never paste the key in a cell, a chat, a screenshot or a repository.** If it leaks, revoke it and create a
new one.

In [ ]:
# --- Startti ADP client ---
import os, requests
STARTTI_BASE_URL = "https://api.startti.ai"

def get_secret(name):
    """Read a secret from Kaggle Secrets, Colab userdata or an environment variable."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)

STARTTI_API_KEY = get_secret("STARTTI_API_KEY")
_STARTTI_HINTS = {401: "invalid or revoked API key", 402: "plan limit reached",
                  403: "the key cannot access this agent", 404: "agent not found — check AGENT_ID",
                  409: "the agent is not published — publish it in ADP first"}

def startti_run(agent_id, prompt, session_key=None, timeout_s=120):
    """Send one message to a published Startti agent and return its text answer."""
    if not STARTTI_API_KEY:
        raise RuntimeError("STARTTI_API_KEY not found. Kaggle: Add-ons → Secrets. Colab: Secrets (key icon). "
                           "On your own computer: set the STARTTI_API_KEY environment variable before starting Jupyter.")
    body = {"agentId": agent_id, "prompt": prompt}
    if session_key:
        body["sessionKey"] = session_key
    r = requests.post(f"{STARTTI_BASE_URL}/v1/run", json=body, timeout=timeout_s,
                      headers={"Authorization": f"Bearer {STARTTI_API_KEY}"})
    if r.status_code >= 400:
        try:
            err = r.json()
        except ValueError:
            err = {"message": r.text[:300]}
        hint = _STARTTI_HINTS.get(r.status_code, "")
        raise RuntimeError(f"Startti API {r.status_code} ({hint}): {err.get('code', '')} {err.get('message', err)}")
    out = r.json().get("output", {})
    if out.get("errorText"):
        raise RuntimeError(f"Agent error: {out['errorText']}")
    if out.get("suspended"):
        questions = "; ".join(q.get("text", "") for q in out.get("suspendQuestions", []))
        return f"[AGENT ASKED] {questions}"
    return out.get("text", "")

print("Startti key found ✅" if STARTTI_API_KEY else "No STARTTI_API_KEY — Startti cells will be skipped.")
# --- end of Startti client ---

`startti_run(AGENT_ID, prompt, session_key=None)` sends `POST https://api.startti.ai/v1/run` with your key as a
Bearer token. Reusing the same `session_key` continues a conversation; without it, every call starts a new one.

In [ ]:
AGENT_ID = ""  # ✏️ paste your agent's ID here (the ID is not secret; the API key is)

STARTTI_READY = bool(STARTTI_API_KEY and AGENT_ID)
TEST_PROMPTS = ["My new card hasn't arrived yet. What can I do?",
                "Can you recommend a good restaurant near the office?",
                "Perdí mi tarjeta, ¿qué hago?"]
STARTTI_HELP = (
    "Startti is not configured yet, so the Startti calls are skipped. To enable them:\n"
    "  1. Create, test and PUBLISH your agent in ADP (steps above).\n"
    "  2. Save your adp_… key as the Kaggle secret STARTTI_API_KEY and re-run the client cell.\n"
    "  3. Paste the agent ID in AGENT_ID and re-run this cell.\n"
    "Guide: https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/docs/configuracion-startti.md")

if STARTTI_API_KEY and AGENT_ID:
    for prompt in TEST_PROMPTS:
        t0 = time.perf_counter()
        answer = startti_run(AGENT_ID, prompt)
        print(f"👤 {prompt}\n🤖 {answer}\n   ⏱️ {time.perf_counter() - t0:.1f} s\n")
else:
    print(STARTTI_HELP)

In [ ]:
# A two-turn conversation: the same session_key keeps the context
if STARTTI_API_KEY and AGENT_ID:
    print(startti_run(AGENT_ID, "I want to block my card", session_key="s06-lab-01"))
    print(startti_run(AGENT_ID, "It's the credit card", session_key="s06-lab-01"))
else:
    print("Skipped: Startti is not configured (see the previous cell).")

**Wrap the agent in a chat interface.** `gr.ChatInterface(fn)` builds a chat UI around any function
`fn(message, history) -> str`. If Startti is not configured, the same interface uses a **Python fallback bot**:
our classifier + threshold + a few canned answers. That fallback is also the course's no-account alternative.

In [ ]:
import uuid

SENSITIVE_INTENTS = {"lost_or_stolen_card", "compromised_card", "card_payment_not_recognised"}
CANNED = {  # answers written and approved by the (fictional) support team
    "card_arrival": "New cards usually take a few working days to arrive. You can follow the delivery status "
                    "in the app. If it is later than the date shown there, I can connect you with an advisor.",
    "activate_my_card": "To activate your card, open the app, go to your card and choose the activation option. "
                        "You will need the card with you.",
    "change_pin": "You can change your PIN from the card settings in the app. Never share your PIN with anyone, "
                  "including us.",
    "exchange_rate": "The exchange rate is shown in the app before you confirm each payment or exchange.",
    "transfer_timing": "Transfers between Andes Bank accounts are usually instant; transfers to other banks can "
                       "take longer. You can check the status in your transaction history.",
}
HANDOFF_TEXT = ("I am not sure I can answer that correctly, so I am passing your message to a human advisor. "
                "You can also rephrase your question.")


SENSITIVE_MIN_PROB = 0.20  # a security intent this likely in the top-3 is enough to skip automation


def python_bot(message, threshold=BEST_THRESHOLD):
    """No-account alternative: classifier + threshold + canned answers + human handoff."""
    top = predict_intent(message, k=3)
    intent, conf = top[0]
    flagged = [name for name, p in top if name in SENSITIVE_INTENTS and p >= SENSITIVE_MIN_PROB]
    if flagged:  # security first, whatever the threshold
        return f"🔒 This may be a security case ({flagged[0]}). I am connecting you with an advisor right now."
    if conf < threshold or intent not in CANNED:
        return HANDOFF_TEXT
    return f"{CANNED[intent]}\n\n_(Virtual assistant · topic `{intent}` · confidence {conf:.0%})_"


_chat = {"session_key": None}


def chat_fn(message, history):
    """One chat turn: Startti agent if configured, otherwise the Python bot."""
    if not history:  # a new conversation starts: new session key
        _chat["session_key"] = f"s06-lab-{uuid.uuid4().hex[:8]}"
    if STARTTI_READY:
        try:
            return startti_run(AGENT_ID, message, session_key=_chat["session_key"])
        except Exception as exc:  # graceful failure: explain and offer a way forward
            print("Startti error:", exc)
            return "Sorry, the assistant is not available right now. A human advisor will contact you."
    return python_bot(message)


for prompt in TEST_PROMPTS:
    print(f"👤 {prompt}\n🤖 {chat_fn(prompt, [])}\n")

chat_demo = gr.ChatInterface(
    chat_fn,
    title="Andes Bank · virtual assistant (prototype)",
    description=("AI assistant for card and transfer questions. It can make mistakes: for anything urgent or "
                 "about your security, ask for a human advisor."),
    examples=["My new card hasn't arrived yet", "How do I change my PIN?", "Someone used my card"],
)

if FAST_DEV_RUN:
    print("Fast dev run: the chat app is built but not launched.")
else:
    chat_demo.launch(share=True)

**Code or no-code?** Compare the two assistants you just built.

| | Python + Gradio | Startti agent |
|---|---|---|
| Where the logic lives | In your code: model, threshold, canned answers | On the canvas: nodes and system instructions |
| Who can change it | Someone who writes Python | Anyone trained on the platform; developers call it by API |
| Languages | Only what the training data covers (English here) | Whatever the chosen language model handles |
| Control of handoff | Exact and testable (threshold, sensitive intents) | Written as instructions: must be tested with conversations |
| Latency and cost | Milliseconds on a CPU; you pay for hosting | Seconds per call; platform plan and model usage |
| Evaluation and logs | You build them (Part 3) | Use the platform's tools and your own test set (Session 07) |

**✋ Checkpoint.** How did each assistant handle the Spanish message and the restaurant question? Which one would
you show to Andes Bank's customers first, and which one to its support team?

**🧪 Try it.** Add a rule to your agent's system instructions (for example: "If the customer mentions fraud,
answer only: *I am connecting you with an advisor*"). Save it, make sure the agent is published, and re-run the
test cell. Did the agent follow the rule every time?

<details><summary>Show a solution</summary>

Test the rule with several phrasings, not just one:

```python
if STARTTI_API_KEY and AGENT_ID:
    for p in ["Someone used my card without permission", "I think this payment is fraud", "Me robaron la tarjeta"]:
        print(p, "→", startti_run(AGENT_ID, p))
```

Instructions steer a language model but do not guarantee its behaviour: that is why a rule that must never fail
(security handoff) is often enforced **outside** the model too, as in `SENSITIVE_INTENTS` of the Python bot.
In Session 07 you will evaluate agents systematically with a set of test conversations.
</details>

## Wrap-up

- A product is more than a model: user, job to be done, inputs/outputs, constraints, logs and feedback.
- A confidence threshold turns a classifier into a **human-in-the-loop** system; choose it on validation data
  by expected cost, and check calibration before trusting the theoretical rule.
- Gradio `Blocks` gives you a working prototype in one cell; a Startti agent gives you a published assistant
  with an API. Both need the same UX care: expectations, confidence, graceful failure and a way to a human.

**Next: the Session Challenge** (`challenge.ipynb`): your own customer-service case and unit costs, the threshold
of minimum expected cost, your Gradio app, a Startti (or Python) assistant and a code vs no-code decision.

**Before Session 07:** your Startti API key must be saved in Kaggle Secrets as `STARTTI_API_KEY` and the client
cell must print `Startti key found ✅`. If you prefer not to use Startti, the Python alternative is enough.